# ILGA Sponsor Report PDF Scraper: Woman × General Assembly Counts

In [ ]:
import re
import time
import hashlib
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote
from difflib import SequenceMatcher

import pandas as pd
import requests

# Install PyMuPDF if needed. Used to extract text from PDFs.
try:
    import fitz  # PyMuPDF
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pymupdf"])
    import fitz

CAWP_CSV = "illinois.csv"
OUTPUT_DIR = Path("output")
CACHE_DIR = Path("cache_ilga_reports")
OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

BASE = "https://www.ilga.gov"
REQUEST_DELAY = 0.25
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; student research project; respectful scraping)"
}

# ILGA legislation records are most useful/reliable from the 90th GA onward.
MIN_GA = 90
MAX_GA = 104

TARGET_MIN_ROWS = 500
TARGET_MAX_ROWS = 1000



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


## 2. Load and clean CAWP Illinois women

In [ ]:
cawp_raw = pd.read_csv(CAWP_CSV)
print("Raw CAWP shape:", cawp_raw.shape)
print("Columns:", list(cawp_raw.columns))

cawp = cawp_raw.copy()

# Keep Illinois rows only
state_cols = [c for c in cawp.columns if c.lower() in {"state", "st", "state name"}]
if state_cols:
    state_col = state_cols[0]
    cawp = cawp[cawp[state_col].astype(str).str.contains("Illinois|IL", case=False, na=False)].copy()

print("Rows after Illinois filter:", cawp.shape)


Raw CAWP shape: (2742, 11)
Columns: ['ID', 'Years Served', 'First Name', 'Middle Name', 'Last Name', 'party', 'Level', 'Position', 'state', 'District', 'race_ethnicity']
Rows after Illinois filter: (2742, 11)


In [3]:
def pick_first_existing(row, candidates):
    for col in candidates:
        if col in row.index and pd.notna(row[col]) and str(row[col]).strip():
            return str(row[col]).strip()
    return ""


def cawp_full_name(row) -> str:
    full = pick_first_existing(row, ["Name", "Full Name", "full_name", "name", "Legislator", "Member"])
    if full:
        return full

    first = pick_first_existing(row, ["First Name", "First", "first_name"])
    middle = pick_first_existing(row, ["Middle Name", "Middle", "middle_name"])
    last = pick_first_existing(row, ["Last Name", "Last", "last_name", "Surname"])
    return " ".join(x for x in [first, middle, last] if x).strip()


def normalize_name(name: str) -> str:
    name = str(name).lower()
    name = re.sub(r"\([^)]*\)", " ", name)
    name = re.sub(r"\b(rep|representative|sen|senator|hon|honorable)\b", " ", name)
    name = re.sub(r"[^a-z\s-]", " ", name)
    name = re.sub(r"\b[a-z]\b", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def last_name_key(name: str) -> str:
    parts = normalize_name(name).split()
    return parts[-1] if parts else ""

# Use a stable person id if one exists; otherwise create one.
possible_id_cols = ["ID", "id", "person_id", "CAWPID", "cawp_id"]
person_id_col = next((c for c in possible_id_cols if c in cawp.columns), None)
if person_id_col is None:
    cawp["cawp_person_id"] = range(1, len(cawp) + 1)
    person_id_col = "cawp_person_id"

cawp["cawp_name"] = cawp.apply(cawp_full_name, axis=1)
cawp["cawp_match_key"] = cawp["cawp_name"].apply(normalize_name)
cawp["cawp_last_name"] = cawp["cawp_name"].apply(last_name_key)

# One row per CAWP person for matching.
cawp_people = (
    cawp.dropna(subset=["cawp_name"])
        .drop_duplicates(subset=[person_id_col, "cawp_match_key"])
        [[person_id_col, "cawp_name", "cawp_match_key", "cawp_last_name"]]
        .copy()
)

print("Unique CAWP people for matching:", len(cawp_people))
display(cawp_people.head())


Unique CAWP people for matching: 279


,ID,cawp_name,cawp_match_key,cawp_last_name
0,12077h,Ethel Skyles Alexander,ethel skyles alexander,alexander
14,214519k,Pamela J. Althoff,pamela althoff,althoff
30,477852k,Carol Ammons,carol ammons,ammons
42,724507k,Dagmara Avelar,dagmara avelar,avelar
48,12449c,Cheryl Axley,cheryl axley,axley


In [4]:
def ordinal(n: int) -> str:
    if 10 <= n % 100 <= 20:
        suffix = "th"
    else:
        suffix = {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suffix}"


def report_url_candidates(ga: int, chamber: str):
    chamber_title = "House" if chamber.lower().startswith("h") else "Senate"
    file_name = f"Sponsor Bill Listing By Name-{chamber_title}.pdf"
    encoded_name = quote(file_name)
    prefixed_name = quote(f"{ordinal(ga)}{file_name}")

    urls = []

    # Historical files usually use prefixes like 101stSponsor Bill Listing By Name-House.pdf.
    urls.append(f"{BASE}/documents/reports/static/{prefixed_name}")

    # Current GA often uses the unprefixed static filename.
    if ga == MAX_GA:
        urls.append(f"{BASE}/documents/reports/static/{encoded_name}")

    return urls


def cache_file_for_url(url: str, suffix: str) -> Path:
    key = hashlib.md5(url.encode("utf-8")).hexdigest()
    return CACHE_DIR / f"{key}{suffix}"


def download_pdf_cached(url: str, force_refresh: bool = False) -> Path | None:
    path = cache_file_for_url(url, ".pdf")
    if path.exists() and path.stat().st_size > 1000 and not force_refresh:
        return path

    try:
        time.sleep(REQUEST_DELAY)
        r = requests.get(url, headers=HEADERS, timeout=45)
        if r.status_code != 200 or not r.content.startswith(b"%PDF"):
            return None
        path.write_bytes(r.content)
        return path
    except Exception as e:
        print("Download failed:", url, e)
        return None


def extract_pdf_text_cached(pdf_path: Path) -> str:
    text_path = pdf_path.with_suffix(".txt")
    if text_path.exists() and text_path.stat().st_size > 100:
        return text_path.read_text(encoding="utf-8", errors="ignore")

    doc = fitz.open(pdf_path)
    parts = []
    for page in doc:
        parts.append(page.get_text("text"))
    text = "\n".join(parts)
    text_path.write_text(text, encoding="utf-8", errors="ignore")
    return text


def fetch_report_text(ga: int, chamber: str):
    for url in report_url_candidates(ga, chamber):
        pdf_path = download_pdf_cached(url)
        if pdf_path is not None:
            text = extract_pdf_text_cached(pdf_path)
            return {
                "general_assembly": ga,
                "sponsor_chamber": "House" if chamber.lower().startswith("h") else "Senate",
                "url": url,
                "pdf_path": str(pdf_path),
                "text": text,
                "status": "ok",
            }

    return {
        "general_assembly": ga,
        "sponsor_chamber": "House" if chamber.lower().startswith("h") else "Senate",
        "url": None,
        "pdf_path": None,
        "text": "",
        "status": "missing",
    }


In [5]:
report_records = []
for ga in range(MIN_GA, MAX_GA + 1):
    for chamber in ["House", "Senate"]:
        rec = fetch_report_text(ga, chamber)
        report_records.append({k: v for k, v in rec.items() if k != "text"})
        print(ga, chamber, rec["status"], rec["url"])

reports_status = pd.DataFrame(report_records)
reports_status.to_csv(OUTPUT_DIR / "ilga_sponsor_report_pdf_download_status.csv", index=False)
display(reports_status)


90 House missing None
90 Senate missing None
91 House missing None
91 Senate missing None
92 House missing None
92 Senate missing None
93 House ok https://www.ilga.gov/documents/reports/static/93rdSponsor%20Bill%20Listing%20By%20Name-House.pdf
93 Senate ok https://www.ilga.gov/documents/reports/static/93rdSponsor%20Bill%20Listing%20By%20Name-Senate.pdf
94 House ok https://www.ilga.gov/documents/reports/static/94thSponsor%20Bill%20Listing%20By%20Name-House.pdf
94 Senate ok https://www.ilga.gov/documents/reports/static/94thSponsor%20Bill%20Listing%20By%20Name-Senate.pdf
95 House ok https://www.ilga.gov/documents/reports/static/95thSponsor%20Bill%20Listing%20By%20Name-House.pdf
95 Senate ok https://www.ilga.gov/documents/reports/static/95thSponsor%20Bill%20Listing%20By%20Name-Senate.pdf
96 House ok https://www.ilga.gov/documents/reports/static/96thSponsor%20Bill%20Listing%20By%20Name-House.pdf
96 Senate ok https://www.ilga.gov/documents/reports/static/96thSponsor%20Bill%20Listing%20By%20N

,general_assembly,sponsor_chamber,url,pdf_path,status
0,90,House,None,None,missing
1,90,Senate,None,None,missing
2,91,House,None,None,missing
3,91,Senate,None,None,missing
4,92,House,None,None,missing
5,92,Senate,None,None,missing
6,93,House,https://www.ilga.gov/documents/reports/static/...,cache_ilga_reports/e63ad54b890339b87a0e8dd826b...,ok
7,93,Senate,https://www.ilga.gov/documents/reports/static/...,cache_ilga_reports/46e59085522701b753733556264...,ok
8,94,House,https://www.ilga.gov/documents/reports/static/...,cache_ilga_reports/e1e38e32380399620de9c3f4b6c...,ok
9,94,Senate,https://www.ilga.gov/documents/reports/static/...,cache_ilga_reports/d533e09d0a0ad4307141052a7b3...,ok


## 4. Parse sponsor bill rows from the PDFs

In [6]:
SPONSOR_TYPES = [
    "Alt. Chief Sponsor",
    "Alt. Co-Sponsor",
    "Chief Co-Sponsor",
    "Chief Sponsor",
    "Co-Sponsor",
]

SPONSOR_TYPE_RE = re.compile(
    r"\b(Alt\. Chief Sponsor|Alt\. Co-Sponsor|Chief Co-Sponsor|Chief Sponsor|Co-Sponsor)\b"
)

BILL_START_RE = re.compile(
    r"^(HB|SB|HR|SR|HJR|SJR|HJRCA|SJRCA)\s*0*(\d+)\b\s*(.*)$",
    re.IGNORECASE,
)

NOISE_PREFIXES = (
    "Bill Number", "Number Short", "Bill Last", "Type", "Legislative Information System",
    "Page:", "Last Action", "Last Act", "Location", "Date", "Act Date"
)


def normalize_bill_number(prefix: str, num: str) -> str:
    return f"{prefix.upper()}{int(num):04d}"


def is_noise_line(line: str) -> bool:
    s = line.strip()
    if not s:
        return True
    if any(s.startswith(p) for p in NOISE_PREFIXES):
        return True
    if re.match(r"^\d{1,2}/\d{1,2}/\d{2,4}", s):
        return True
    return False


def split_type_from_text(text: str):
    m = SPONSOR_TYPE_RE.search(text)
    if not m:
        return None
    title = text[:m.start()].strip()
    sponsor_type = m.group(1).strip()
    rest = text[m.end():].strip()

    # Location is often the first H/S after sponsor type.
    loc_match = re.match(r"^([HS])\b\s*(.*)$", rest)
    location = loc_match.group(1) if loc_match else ""
    last_action = loc_match.group(2).strip() if loc_match else rest

    return title, sponsor_type, location, last_action


def parse_sponsor_report_text(text: str, ga: int, sponsor_chamber: str):
    rows = []
    current_sponsor = None
    current_bill = None

    def finalize_bill():
        nonlocal current_bill
        if current_bill is not None and current_bill.get("sponsorship_type"):
            title = " ".join(current_bill.get("title_parts", []))
            title = re.sub(r"\s+", " ", title).strip()
            row = {
                "general_assembly": ga,
                "sponsor_chamber": sponsor_chamber,
                "sponsor_name": current_sponsor,
                "bill_number": current_bill["bill_number"],
                "short_title": title,
                "sponsorship_type": current_bill["sponsorship_type"],
                "bill_location": current_bill.get("bill_location", ""),
                "last_action_fragment": current_bill.get("last_action_fragment", ""),
            }
            rows.append(row)
        current_bill = None

    for raw_line in text.splitlines():
        line = re.sub(r"\s+", " ", raw_line).strip()
        if not line:
            continue

        # New sponsor section.
        if line.startswith("Sponsor Legislation Listing For "):
            finalize_bill()
            current_sponsor = line.replace("Sponsor Legislation Listing For ", "").strip()
            continue

        if current_sponsor is None or is_noise_line(line):
            continue

        bill_match = BILL_START_RE.match(line)
        if bill_match:
            finalize_bill()
            prefix, num, rest = bill_match.groups()
            current_bill = {
                "bill_number": normalize_bill_number(prefix, num),
                "title_parts": [],
                "sponsorship_type": None,
                "bill_location": "",
                "last_action_fragment": "",
            }

            split = split_type_from_text(rest)
            if split:
                title, sponsor_type, location, last_action = split
                current_bill["title_parts"].append(title)
                current_bill["sponsorship_type"] = sponsor_type
                current_bill["bill_location"] = location
                current_bill["last_action_fragment"] = last_action
                finalize_bill()
            else:
                if rest.strip():
                    current_bill["title_parts"].append(rest.strip())
            continue

        # Continuation line for title and/or type.
        if current_bill is not None:
            split = split_type_from_text(line)
            if split:
                title, sponsor_type, location, last_action = split
                if title:
                    current_bill["title_parts"].append(title)
                current_bill["sponsorship_type"] = sponsor_type
                current_bill["bill_location"] = location
                current_bill["last_action_fragment"] = last_action
                finalize_bill()
            else:
                # Avoid adding obvious footer/header junk.
                if not is_noise_line(line):
                    current_bill["title_parts"].append(line)

    finalize_bill()
    return rows


In [7]:
all_bill_rows = []
report_fetch_records = []

for ga in range(MIN_GA, MAX_GA + 1):
    for chamber in ["House", "Senate"]:
        rec = fetch_report_text(ga, chamber)
        report_fetch_records.append({k: v for k, v in rec.items() if k != "text"})
        if rec["status"] != "ok":
            continue

        rows = parse_sponsor_report_text(rec["text"], ga, rec["sponsor_chamber"])
        all_bill_rows.extend(rows)
        print(f"Parsed GA {ga} {chamber}: {len(rows):,} bill-sponsor rows")

bill_sponsor_rows = pd.DataFrame(all_bill_rows)

if bill_sponsor_rows.empty:
    raise ValueError("No bill-sponsor rows were parsed. Check output/ilga_sponsor_report_pdf_download_status.csv")

bill_sponsor_rows["sponsor_match_key"] = bill_sponsor_rows["sponsor_name"].apply(normalize_name)
bill_sponsor_rows["sponsor_last_name"] = bill_sponsor_rows["sponsor_name"].apply(last_name_key)

bill_sponsor_rows = bill_sponsor_rows.drop_duplicates(
    subset=["general_assembly", "sponsor_chamber", "sponsor_name", "bill_number", "sponsorship_type"]
).copy()

bill_sponsor_rows.to_csv(OUTPUT_DIR / "ilga_report_bill_sponsor_rows_all_people.csv", index=False)

print("Total parsed bill-sponsor rows:", len(bill_sponsor_rows))
print("Unique sponsors:", bill_sponsor_rows["sponsor_name"].nunique())
display(bill_sponsor_rows.head())


Parsed GA 93 House: 31,122 bill-sponsor rows
Parsed GA 93 Senate: 13,343 bill-sponsor rows
Parsed GA 94 House: 29,304 bill-sponsor rows
Parsed GA 94 Senate: 11,528 bill-sponsor rows
Parsed GA 95 House: 34,590 bill-sponsor rows
Parsed GA 95 Senate: 12,275 bill-sponsor rows
Parsed GA 96 House: 31,625 bill-sponsor rows
Parsed GA 96 Senate: 12,120 bill-sponsor rows
Parsed GA 97 House: 23,071 bill-sponsor rows
Parsed GA 97 Senate: 10,081 bill-sponsor rows
Parsed GA 98 House: 25,358 bill-sponsor rows
Parsed GA 98 Senate: 11,194 bill-sponsor rows
Parsed GA 99 House: 27,744 bill-sponsor rows
Parsed GA 99 Senate: 12,785 bill-sponsor rows
Parsed GA 100 House: 29,139 bill-sponsor rows
Parsed GA 100 Senate: 14,333 bill-sponsor rows
Parsed GA 101 House: 24,778 bill-sponsor rows
Parsed GA 101 Senate: 12,674 bill-sponsor rows
Parsed GA 102 House: 28,027 bill-sponsor rows
Parsed GA 102 Senate: 15,269 bill-sponsor rows
Parsed GA 103 House: 27,800 bill-sponsor rows
Parsed GA 103 Senate: 15,449 bill-spon

,general_assembly,sponsor_chamber,sponsor_name,bill_number,short_title,sponsorship_type,bill_location,last_action_fragment,sponsor_match_key,sponsor_last_name
0,93,House,Edward J. Acevedo,HB0020,H/ED-TEACH ILL SCHOLARSHIP,Co-Sponsor,,,edward acevedo,acevedo
1,93,House,Edward J. Acevedo,HB0037,CHILDCARE REPORT-HUMAN SERV,Co-Sponsor,,,edward acevedo,acevedo
2,93,House,Edward J. Acevedo,HB0041,DCFS-MENTAL HEALTH SERVICES,Co-Sponsor,,,edward acevedo,acevedo
3,93,House,Edward J. Acevedo,HB0043,FITNESS FACILITY-DEFIBRILL ATOR,Co-Sponsor,,,edward acevedo,acevedo
4,93,House,Edward J. Acevedo,HB0050,ELDER CARE SAVINGS FUND ACT,Co-Sponsor,,,edward acevedo,acevedo


## 5. Match CAWP women to ILGA sponsor names

In [ ]:
def name_similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, normalize_name(a), normalize_name(b)).ratio()


# Format: "CAWP name": "ILGA sponsor name"
MANUAL_MATCH_FIXES = {
    # Example:
    # "Cynthia Soto": "Cynthia Soto",
}

sponsor_people = (
    bill_sponsor_rows[["sponsor_name", "sponsor_match_key", "sponsor_last_name"]]
    .drop_duplicates()
    .copy()
)

matches = []
for _, c_row in cawp_people.iterrows():
    c_name = c_row["cawp_name"]
    c_key = c_row["cawp_match_key"]
    c_last = c_row["cawp_last_name"]

    if c_name in MANUAL_MATCH_FIXES:
        forced = MANUAL_MATCH_FIXES[c_name]
        candidates = sponsor_people[sponsor_people["sponsor_name"] == forced].copy()
    else:
        candidates = sponsor_people[sponsor_people["sponsor_last_name"] == c_last].copy()

    if candidates.empty:
        continue

    candidates["match_score"] = candidates["sponsor_match_key"].apply(lambda x: SequenceMatcher(None, c_key, x).ratio())
    best = candidates.sort_values("match_score", ascending=False).iloc[0]

    if best["match_score"] >= 0.82 or c_name in MANUAL_MATCH_FIXES:
        matches.append({
            person_id_col: c_row[person_id_col],
            "cawp_name": c_name,
            "cawp_match_key": c_key,
            "sponsor_name": best["sponsor_name"],
            "sponsor_match_key": best["sponsor_match_key"],
            "match_score": best["match_score"],
        })

matched_people = pd.DataFrame(matches).drop_duplicates()
matched_people.to_csv(OUTPUT_DIR / "matched_cawp_to_ilga_report_sponsors.csv", index=False)

unmatched_people = cawp_people[~cawp_people[person_id_col].isin(matched_people[person_id_col])].copy()
unmatched_people.to_csv(OUTPUT_DIR / "unmatched_cawp_to_ilga_report_sponsors.csv", index=False)

print("Matched CAWP people:", matched_people[person_id_col].nunique())
print("Unmatched CAWP people:", len(unmatched_people))
display(matched_people.sort_values("match_score").head(20))


Matched CAWP people: 159
Unmatched CAWP people: 120


,ID,cawp_name,cawp_match_key,sponsor_name,sponsor_match_key,match_score
33,646501k,Rachelle Aud Crowe,rachelle aud crowe,Rachelle Crowe,rachelle crowe,0.875
0,214519k,Pamela J. Althoff,pamela althoff,Pamela J. Althoff,pamela althoff,1.000
102,2720c,Patricia Reid Lindner,patricia reid lindner,Patricia Reid Lindner,patricia reid lindner,1.000
103,724349k,Meg Loughran Cappel,meg loughran cappel,Meg Loughran Cappel,meg loughran cappel,1.000
104,3966c,Eileen Lyons,eileen lyons,Eileen Lyons,eileen lyons,1.000
105,560990k,Theresa Mah,theresa mah,Theresa Mah,theresa mah,1.000
106,387729k,Natalie A. Manley,natalie manley,Natalie A. Manley,natalie manley,1.000
107,2713c,Rosemary Mulligan,rosemary mulligan,Rosemary Mulligan,rosemary mulligan,1.000
108,11138c,Ruth Munson,ruth munson,Ruth Munson,ruth munson,1.000
109,555460k,Laura M. Murphy,laura murphy,Laura M. Murphy,laura murphy,1.000


## 6. Keep only matched women and classify topics

In [9]:
women_bill_rows = bill_sponsor_rows.merge(
    matched_people[[person_id_col, "cawp_name", "sponsor_name", "match_score"]],
    on="sponsor_name",
    how="inner",
)

# Topic keywords based on ILGA short titles.
TOPIC_KEYWORDS = {
    "healthcare": [
        "HEALTH", "HEALTH CARE", "MEDICAID", "MEDICARE", "HOSPITAL", "NURSE", "NURSING",
        "MENTAL HEALTH", "MENTAL HLTH", "INS CD-MENTAL", "PUBLIC AID", "DHS", "DHFS",
        "DPH", "IDPH", "MEDICAL", "PHARMACY", "PRESCRIPTION", "SUBSTANCE", "OPIOID",
        "DISABILITY", "DISABILITIES", "AGING", "HOME SERVICES", "BEHAVIORAL", "MATERNAL",
    ],
    "employment": [
        "EMPLOY", "EMPLOYMENT", "LABOR", "WORKFORCE", "WORKER", "WORKERS", "WAGE",
        "PREVAILING WAGE", "UNEMPLOY", "JOB", "HIRING", "WORKPLACE", "PUBLIC EMPLOYEE",
        "PENSION", "PEN CD", "COLLECTIVE BARGAIN", "OCCUPATIONAL", "32 HOUR WORK",
    ],
    "education_children": [
        "EDUC", "EDUCATION", "SCHOOL", "SCH CD", "HIGHER ED", "STUDENT", "TEACHER",
        "TEACH", "CHILD", "CHILDREN", "YOUTH", "JUVENILE", "EARLY LEARNING", "DCFS",
        "DAY CARE", "CHILD CARE", "COLLEGE", "UNIVERSITY", "TUITION", "PUPIL", "K-12",
    ],
}


def classify_topics(title: str):
    upper = str(title).upper()
    topics = []
    for topic, keywords in TOPIC_KEYWORDS.items():
        if any(k in upper for k in keywords):
            topics.append(topic)
    return topics

women_bill_rows["topics"] = women_bill_rows["short_title"].apply(classify_topics)
women_bill_rows["topic_string"] = women_bill_rows["topics"].apply(lambda x: ";".join(x) if x else "uncategorized")

# Chief Sponsor and Alt. Chief Sponsor are treated as primary sponsorship.
# Co-Sponsor, Chief Co-Sponsor, and Alt. Co-Sponsor are treated as co-sponsorship.
women_bill_rows["is_primary"] = women_bill_rows["sponsorship_type"].isin(["Chief Sponsor", "Alt. Chief Sponsor"])
women_bill_rows["is_cosponsor"] = women_bill_rows["sponsorship_type"].isin(["Co-Sponsor", "Chief Co-Sponsor", "Alt. Co-Sponsor"])

women_bill_rows.to_csv(OUTPUT_DIR / "ilga_report_women_bill_rows_evidence.csv", index=False)

print("Matched women bill-sponsor rows:", len(women_bill_rows))
print("Matched women person-GA combos:", women_bill_rows[[person_id_col, "sponsor_chamber", "general_assembly"]].drop_duplicates().shape[0])
display(women_bill_rows.head())


Matched women bill-sponsor rows: 145908
Matched women person-GA combos: 602


,general_assembly,sponsor_chamber,sponsor_name,bill_number,short_title,sponsorship_type,bill_location,last_action_fragment,sponsor_match_key,sponsor_last_name,ID,cawp_name,match_score,topics,topic_string,is_primary,is_cosponsor
0,93,House,Patricia Bailey,HB0014,CRIM CD-CHILD PORN IMAGES,Co-Sponsor,,,patricia bailey,bailey,10122c,Patricia Bailey,1.0,[education_children],education_children,False,True
1,93,House,Patricia Bailey,HB0024,LAW ENFORCEMENT STATISTICS,Co-Sponsor,,,patricia bailey,bailey,10122c,Patricia Bailey,1.0,[],uncategorized,False,True
2,93,House,Patricia Bailey,HB0025,CIV PRO-COURT ORDER-HAZARD,Chief Co-Sponsor,,,patricia bailey,bailey,10122c,Patricia Bailey,1.0,[],uncategorized,False,True
3,93,House,Patricia Bailey,HB0026,TOLL HIGHWAY-NO FINANCL BENEFT,Chief Co-Sponsor,,,patricia bailey,bailey,10122c,Patricia Bailey,1.0,[],uncategorized,False,True
4,93,House,Patricia Bailey,HB0037,CHILDCARE REPORT-HUMAN SERV,Chief Co-Sponsor,,,patricia bailey,bailey,10122c,Patricia Bailey,1.0,[education_children],education_children,False,True


## 7. Summarize to one row per woman × chamber × General Assembly

In [ ]:
def has_topic_list(topics, topic):
    if isinstance(topics, list):
        return topic in topics
    return topic in str(topics).split(";")


def summarize_person_ga(group):
    out = {
        "ilga_all_bills_count": len(group),
        "ilga_primary_bills_count": int(group["is_primary"].sum()),
        "ilga_cosponsored_bills_count": int(group["is_cosponsor"].sum()),
    }

    for topic in TOPIC_KEYWORDS:
        has_topic = group["topics"].apply(lambda x: has_topic_list(x, topic))
        out[f"ilga_{topic}_all_count"] = int(has_topic.sum())
        out[f"ilga_{topic}_primary_count"] = int((has_topic & group["is_primary"]).sum())
        out[f"ilga_{topic}_cosponsored_count"] = int((has_topic & group["is_cosponsor"]).sum())

    return pd.Series(out)

person_ga_counts = (
    women_bill_rows
    .groupby([person_id_col, "cawp_name", "sponsor_name", "sponsor_chamber", "general_assembly"], dropna=False)
    .apply(summarize_person_ga)
    .reset_index()
    .rename(columns={"sponsor_name": "ilga_name", "sponsor_chamber": "chamber"})
    .sort_values(["general_assembly", "chamber", "ilga_name"])
)

# If over the upper bound, keep the most recent General Assemblies until the dataset is <= TARGET_MAX_ROWS,

full_count = len(person_ga_counts)
selected = person_ga_counts.copy()
if full_count > TARGET_MAX_ROWS:
    selected_gas = []
    for ga in sorted(person_ga_counts["general_assembly"].unique(), reverse=True):
        selected_gas.append(ga)
        candidate = person_ga_counts[person_ga_counts["general_assembly"].isin(selected_gas)].copy()
        if len(candidate) >= TARGET_MIN_ROWS:
            selected = candidate
            break

person_ga_counts.to_csv(OUTPUT_DIR / "FULL_ilga_women_person_ga_bill_counts_FROM_REPORTS.csv", index=False)
selected.to_csv(OUTPUT_DIR / "FINAL_ilga_women_person_ga_bill_counts_FROM_REPORTS.csv", index=False)

print("Full woman-GA rows:", len(person_ga_counts))
print("Final selected rows:", len(selected))
print("General Assemblies in final:", sorted(selected["general_assembly"].unique()))
display(selected.head(30))


Full woman-GA rows: 602
Final selected rows: 602
General Assemblies in final: [np.int64(93), np.int64(94), np.int64(95), np.int64(96), np.int64(97), np.int64(98), np.int64(99), np.int64(100), np.int64(101), np.int64(102), np.int64(103), np.int64(104)]


/var/folders/wz/7mdpr8553cx8zbp6typ185fr0000gn/T/ipykernel_68895/1070071533.py:25: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_person_ga)


,ID,cawp_name,ilga_name,chamber,general_assembly,ilga_all_bills_count,ilga_primary_bills_count,ilga_cosponsored_bills_count,ilga_healthcare_all_count,ilga_healthcare_primary_count,ilga_healthcare_cosponsored_count,ilga_employment_all_count,ilga_employment_primary_count,ilga_employment_cosponsored_count,ilga_education_children_all_count,ilga_education_children_primary_count,ilga_education_children_cosponsored_count
594,9052c,Annazette R. Collins,Annazette Collins,House,93,168,29,139,16,0,16,5,1,4,34,3,31
104,194309k,Barbara Flynn Currie,Barbara Flynn Currie,House,93,674,118,556,40,0,40,75,1,74,58,7,51
37,11216c,Careen M. Gordon,Careen Gordon,House,93,128,21,107,12,0,12,12,1,11,11,3,8
220,2710c,Carole Pankau,Carole Pankau,House,93,81,25,56,8,0,8,1,1,0,5,0,5
230,2714c,Carolyn H. Krause,Carolyn H. Krause,House,93,174,50,124,22,5,17,8,3,5,23,5,18
365,3962c,Constance A. Howard,Constance A. Howard,House,93,249,84,165,32,9,23,11,5,6,34,5,29
127,208356k,Cynthia Soto,Cynthia Soto,House,93,247,39,208,36,7,29,14,2,12,35,9,26
24,10713c,Deborah L. Graham,Deborah L. Graham,House,93,249,29,220,31,2,29,10,0,10,44,3,41
370,3966c,Eileen Lyons,Eileen Lyons,House,93,226,43,183,34,1,33,5,1,4,40,11,29
138,212690k,Elaine Nekritz,Elaine Nekritz,House,93,239,50,189,31,3,28,13,0,13,37,5,32


## 8. Final checks

In [11]:
final_df = pd.read_csv(OUTPUT_DIR / "FINAL_ilga_women_person_ga_bill_counts_FROM_REPORTS.csv")

print("Final rows:", len(final_df))
print("Unique women:", final_df["ilga_name"].nunique())
print("GA range:", final_df["general_assembly"].min(), "to", final_df["general_assembly"].max())
print("Total bills counted:", final_df["ilga_all_bills_count"].sum())

count_cols = [c for c in final_df.columns if c.startswith("ilga_") and c.endswith("_count")]
display(final_df[count_cols].describe().T)

display(
    final_df.groupby("general_assembly")
    .agg(rows=("ilga_name", "count"), women=("ilga_name", "nunique"), bills=("ilga_all_bills_count", "sum"))
    .reset_index()
)


Final rows: 602
Unique women: 158
GA range: 93 to 104
Total bills counted: 145908


,count,mean,std,min,25%,50%,75%,max
ilga_all_bills_count,602.0,242.372093,133.591853,1.0,166.00,231.0,304.75,1415.0
ilga_primary_bills_count,602.0,61.983389,54.438532,0.0,32.25,52.0,78.00,642.0
ilga_cosponsored_bills_count,602.0,180.388704,111.779190,0.0,113.00,165.0,237.00,1240.0
ilga_healthcare_all_count,602.0,27.906977,19.201046,0.0,14.00,25.0,38.00,114.0
ilga_healthcare_primary_count,602.0,5.991694,7.173736,0.0,1.00,4.0,8.00,60.0
ilga_healthcare_cosponsored_count,602.0,21.915282,15.167982,0.0,10.00,20.0,31.00,84.0
ilga_employment_all_count,602.0,11.892027,9.351340,0.0,6.00,11.0,15.00,75.0
ilga_employment_primary_count,602.0,2.676080,4.735225,0.0,0.25,1.5,3.00,63.0
ilga_employment_cosponsored_count,602.0,9.215947,7.874423,0.0,4.00,8.0,12.00,74.0
ilga_education_children_all_count,602.0,34.822259,21.423469,0.0,19.25,31.0,47.00,138.0


,general_assembly,rows,women,bills
0,93,48,47,11942
1,94,47,47,11898
2,95,44,44,13389
3,96,44,44,11750
4,97,46,45,8385
5,98,49,49,9892
6,99,43,43,10483
7,100,48,48,12825
8,101,56,54,10940
9,102,60,60,15284


## Audit

In [12]:
final_df = pd.read_csv("output/FINAL_ilga_women_person_ga_bill_counts_FROM_REPORTS.csv")
evidence = pd.read_csv("output/ilga_report_women_bill_rows_evidence.csv")

display(
    final_df.sort_values("ilga_all_bills_count", ascending=False)
    [[
        "cawp_name",
        "ilga_name",
        "chamber",
        "general_assembly",
        "ilga_all_bills_count",
        "ilga_primary_bills_count",
        "ilga_cosponsored_bills_count",
        "ilga_healthcare_all_count",
        "ilga_employment_all_count",
        "ilga_education_children_all_count",
    ]]
    .head(20)
)

,cawp_name,ilga_name,chamber,general_assembly,ilga_all_bills_count,ilga_primary_bills_count,ilga_cosponsored_bills_count,ilga_healthcare_all_count,ilga_employment_all_count,ilga_education_children_all_count
140,Barbara Flynn Currie,Barbara Flynn Currie,House,96,1415,175,1240,64,64,88
49,Barbara Flynn Currie,Barbara Flynn Currie,House,94,1010,109,901,44,73,65
96,Barbara Flynn Currie,Barbara Flynn Currie,House,95,971,110,861,59,69,78
318,Pamela J. Althoff,Pamela J. Althoff,Senate,99,831,642,189,24,24,29
1,Barbara Flynn Currie,Barbara Flynn Currie,House,93,674,118,556,40,75,58
110,Linda Chapa LaVia,Linda Chapa LaVia,House,95,635,113,522,60,16,113
102,Elizabeth Coulson,Elizabeth Coulson,House,95,634,176,458,110,10,92
114,Monique D. Davis,Monique D. Davis,House,95,611,64,547,26,19,79
310,Christine Radogno,Christine Radogno,Senate,99,608,582,26,20,64,33
275,Pamela J. Althoff,Pamela J. Althoff,Senate,98,604,440,164,18,21,21


In [13]:
evidence["bill_type"] = evidence["bill_number"].str.extract(r"^([A-Z]+)")

display(
    evidence.groupby("bill_type")
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
)

,bill_type,rows
0,HB,79440
4,SB,43075
3,HR,13501
7,SR,4880
1,HJR,2897
5,SJR,1128
2,HJRCA,690
6,SJRCA,297


In [14]:
evidence = pd.read_csv("output/ilga_report_women_bill_rows_evidence.csv")

evidence["bill_type"] = evidence["bill_number"].str.extract(r"^([A-Z]+)")
evidence_bills_only = evidence[evidence["bill_type"].isin(["HB", "SB"])].copy()

# Collapse to one row per woman-GA-bill.
# This prevents the same bill from being counted twice for the same person.
group_cols = [
    "ID",
    "cawp_name",
    "sponsor_name",
    "sponsor_chamber",
    "general_assembly",
    "bill_number",
]

collapsed = (
    evidence_bills_only
    .groupby(group_cols, dropna=False)
    .agg(
        short_title=("short_title", "first"),
        is_primary=("is_primary", "max"),
        is_cosponsor=("is_cosponsor", "max"),
        topic_string=("topic_string", lambda x: ";".join(sorted(set(";".join(x.astype(str)).split(";"))))),
    )
    .reset_index()
)

# If someone is listed as both primary and cosponsor on the same bill, count it as primary.
collapsed["is_cosponsor"] = collapsed["is_cosponsor"] & ~collapsed["is_primary"]

def has_topic_string(s, topic):
    return topic in str(s).split(";")

def summarize_person_ga_strict(group):
    out = {
        "ilga_all_bills_count": len(group),
        "ilga_primary_bills_count": int(group["is_primary"].sum()),
        "ilga_cosponsored_bills_count": int(group["is_cosponsor"].sum()),
    }

    for topic in ["healthcare", "employment", "education_children"]:
        has_topic = group["topic_string"].apply(lambda x: has_topic_string(x, topic))
        out[f"ilga_{topic}_all_count"] = int(has_topic.sum())
        out[f"ilga_{topic}_primary_count"] = int((has_topic & group["is_primary"]).sum())
        out[f"ilga_{topic}_cosponsored_count"] = int((has_topic & group["is_cosponsor"]).sum())

    return pd.Series(out)

person_ga_counts_strict = (
    collapsed
    .groupby(["ID", "cawp_name", "sponsor_name", "sponsor_chamber", "general_assembly"], dropna=False)
    .apply(summarize_person_ga_strict)
    .reset_index()
    .rename(columns={"sponsor_name": "ilga_name", "sponsor_chamber": "chamber"})
    .sort_values(["general_assembly", "chamber", "ilga_name"])
)

person_ga_counts_strict.to_csv(
    "output/FINAL_ilga_women_person_ga_HB_SB_only_counts.csv",
    index=False
)

print("Strict HB/SB-only rows:", len(person_ga_counts_strict))
print("Unique women:", person_ga_counts_strict["ilga_name"].nunique())
print("GA range:", person_ga_counts_strict["general_assembly"].min(), "to", person_ga_counts_strict["general_assembly"].max())

display(person_ga_counts_strict.head(20))
display(person_ga_counts_strict.describe().T)

Strict HB/SB-only rows: 596
Unique women: 157
GA range: 93 to 104


/var/folders/wz/7mdpr8553cx8zbp6typ185fr0000gn/T/ipykernel_68895/3140028614.py:53: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_person_ga_strict)


,ID,cawp_name,ilga_name,chamber,general_assembly,ilga_all_bills_count,ilga_primary_bills_count,ilga_cosponsored_bills_count,ilga_healthcare_all_count,ilga_healthcare_primary_count,ilga_healthcare_cosponsored_count,ilga_employment_all_count,ilga_employment_primary_count,ilga_employment_cosponsored_count,ilga_education_children_all_count,ilga_education_children_primary_count,ilga_education_children_cosponsored_count
588,9052c,Annazette R. Collins,Annazette Collins,House,93,146,27,119,16,0,16,5,1,4,32,3,29
104,194309k,Barbara Flynn Currie,Barbara Flynn Currie,House,93,588,59,529,40,0,40,74,1,73,57,7,50
37,11216c,Careen M. Gordon,Careen Gordon,House,93,108,12,96,11,0,11,10,1,9,10,2,8
220,2710c,Carole Pankau,Carole Pankau,House,93,53,17,36,8,0,8,1,1,0,4,0,4
230,2714c,Carolyn H. Krause,Carolyn H. Krause,House,93,128,35,93,21,5,16,7,2,5,21,5,16
363,3962c,Constance A. Howard,Constance A. Howard,House,93,190,52,138,30,8,22,11,5,6,32,5,27
127,208356k,Cynthia Soto,Cynthia Soto,House,93,217,35,182,35,7,28,11,2,9,35,9,26
24,10713c,Deborah L. Graham,Deborah L. Graham,House,93,220,23,197,30,2,28,10,0,10,44,3,41
368,3966c,Eileen Lyons,Eileen Lyons,House,93,191,39,152,31,1,30,3,0,3,37,11,26
138,212690k,Elaine Nekritz,Elaine Nekritz,House,93,195,44,151,28,3,25,9,0,9,36,5,31


,count,mean,std,min,25%,50%,75%,max
general_assembly,596.0,98.820470,3.520836,93.0,96.00,99.0,102.00,104.0
ilga_all_bills_count,596.0,205.493289,115.059918,1.0,135.75,195.0,258.25,1319.0
ilga_primary_bills_count,596.0,47.300336,38.471713,0.0,25.00,41.0,61.00,574.0
ilga_cosponsored_bills_count,596.0,158.192953,103.567043,0.0,93.00,145.0,208.25,1223.0
ilga_healthcare_all_count,596.0,26.511745,18.087598,0.0,13.00,24.0,36.00,106.0
ilga_healthcare_primary_count,596.0,5.723154,6.832119,0.0,1.00,4.0,7.00,60.0
ilga_healthcare_cosponsored_count,596.0,20.788591,14.324766,0.0,10.00,19.0,29.00,76.0
ilga_employment_all_count,596.0,11.411074,9.199222,0.0,6.00,10.0,15.00,74.0
ilga_employment_primary_count,596.0,2.590604,4.719013,0.0,0.00,1.0,3.00,63.0
ilga_employment_cosponsored_count,596.0,8.820470,7.715290,0.0,4.00,7.0,12.00,73.0


In [15]:
person_ga_counts[person_ga_counts["ilga_all_bills_count"] == 0].shape[0]

0

## Visualizations

In [ ]:
import pandas as pd
import altair as alt
from pathlib import Path

alt.data_transformers.disable_max_rows()

OUTPUT_DIR = Path("output")

strict_path = OUTPUT_DIR / "FINAL_ilga_women_person_ga_HB_SB_only_counts.csv"
reports_path = OUTPUT_DIR / "FINAL_ilga_women_person_ga_bill_counts_FROM_REPORTS.csv"

if strict_path.exists():
    df = pd.read_csv(strict_path)
else:
    df = pd.read_csv(reports_path)

# Make column names consistent 
rename_map = {
    "sponsor_name": "ilga_name",
    "sponsor_chamber": "chamber"
}
df = df.rename(columns=rename_map)

df["general_assembly"] = df["general_assembly"].astype(int)
df["ga_label"] = "GA " + df["general_assembly"].astype(str)

topic_cols = [
    "ilga_healthcare_all_count",
    "ilga_employment_all_count",
    "ilga_education_children_all_count"
]

df["topic_total_count"] = df[topic_cols].sum(axis=1)

pastel_colors = ["#A8DADC", "#F4A6A6", "#CDB4DB", "#B7E4C7", "#FFD6A5", "#BDE0FE"]

def style_chart(chart):
    return (
        chart
        .configure_title(
            font="Georgia",
            fontSize=18,
            anchor="start",
            color="#2F2F2F"
        )
        .configure_axis(
            labelFont="Georgia",
            titleFont="Georgia",
            labelColor="#3A3A3A",
            titleColor="#3A3A3A",
            gridColor="#EAEAEA"
        )
        .configure_legend(
            labelFont="Georgia",
            titleFont="Georgia"
        )
        .configure_view(
            strokeWidth=0
        )
    )

In [17]:
ga_totals = (
    df.groupby("general_assembly", as_index=False)
    .agg(
        total_bills=("ilga_all_bills_count", "sum"),
        primary_bills=("ilga_primary_bills_count", "sum"),
        cosponsored_bills=("ilga_cosponsored_bills_count", "sum"),
        women_count=("ilga_name", "nunique")
    )
)

chart1 = alt.Chart(ga_totals).mark_line(point=True, strokeWidth=3).encode(
    x=alt.X("general_assembly:O", title="General Assembly"),
    y=alt.Y("total_bills:Q", title="Total Bills Sponsored or Co-Sponsored"),
    tooltip=[
        "general_assembly",
        "total_bills",
        "primary_bills",
        "cosponsored_bills",
        "women_count"
    ]
).properties(
    title="Legislative Productivity of Illinois Women Legislators Over Time",
    width=700,
    height=400
)

style_chart(chart1)

alt.Chart(...)

In [25]:
ga_avg = (
    df.groupby("general_assembly", as_index=False)
    .agg(
        avg_bills_per_woman=("ilga_all_bills_count", "mean"),
        avg_primary_per_woman=("ilga_primary_bills_count", "mean"),
        avg_cosponsored_per_woman=("ilga_cosponsored_bills_count", "mean"),
        women_count=("ilga_name", "nunique")
    )
)

chart2 = alt.Chart(ga_avg).mark_bar(cornerRadiusTopLeft=5, cornerRadiusTopRight=5).encode(
    x=alt.X("general_assembly:O", title="General Assembly"),
    y=alt.Y("avg_bills_per_woman:Q", title="Average Bills per Woman"),
    color=alt.value("#A8DADC"),
    tooltip=[
        "general_assembly",
        alt.Tooltip("avg_bills_per_woman:Q", format=".1f"),
        alt.Tooltip("avg_primary_per_woman:Q", format=".1f"),
        alt.Tooltip("avg_cosponsored_per_woman:Q", format=".1f"),
        "women_count"
    ]
).properties(
    title="Average Legislative Productivity per Congresswoman by General Assembly",
    width=700,
    height=400
)

style_chart(chart2)

alt.Chart(...)

In [26]:
topic_by_ga = (
    df.groupby("general_assembly", as_index=False)
    .agg(
        healthcare=("ilga_healthcare_all_count", "sum"),
        employment=("ilga_employment_all_count", "sum"),
        education_children=("ilga_education_children_all_count", "sum")
    )
)

topic_long = topic_by_ga.melt(
    id_vars="general_assembly",
    var_name="policy_area",
    value_name="bill_count"
)

topic_long["policy_area"] = topic_long["policy_area"].replace({
    "healthcare": "Healthcare",
    "employment": "Employment",
    "education_children": "Education / Children"
})

chart3 = alt.Chart(topic_long).mark_line(point=True, strokeWidth=3).encode(
    x=alt.X("general_assembly:O", title="General Assembly"),
    y=alt.Y("bill_count:Q", title="Total Bills"),
    color=alt.Color(
        "policy_area:N",
        title="Policy Area",
        scale=alt.Scale(range=["#F4A6A6", "#A8DADC", "#CDB4DB"])
    ),
    tooltip=["general_assembly", "policy_area", "bill_count"]
).properties(
    title="Policy Issue Trends Among Illinois Congresswoman Legislators",
    width=750,
    height=420
)

style_chart(chart3)

alt.Chart(...)

In [21]:
chamber_avg = (
    df.groupby(["general_assembly", "chamber"], as_index=False)
    .agg(
        avg_bills=("ilga_all_bills_count", "mean"),
        avg_primary=("ilga_primary_bills_count", "mean"),
        women_count=("ilga_name", "nunique")
    )
)

chart6 = alt.Chart(chamber_avg).mark_line(point=True, strokeWidth=3).encode(
    x=alt.X("general_assembly:O", title="General Assembly"),
    y=alt.Y("avg_bills:Q", title="Average Bills per Woman"),
    color=alt.Color(
        "chamber:N",
        title="Chamber",
        scale=alt.Scale(range=["#A8DADC", "#FFD6A5"])
    ),
    tooltip=[
        "general_assembly",
        "chamber",
        alt.Tooltip("avg_bills:Q", format=".1f"),
        alt.Tooltip("avg_primary:Q", format=".1f"),
        "women_count"
    ]
).properties(
    title="Average Legislative Productivity by Chamber",
    width=750,
    height=420
)

style_chart(chart6)

alt.Chart(...)

In [27]:
top_women = (
    df.groupby("ilga_name", as_index=False)
    .agg(
        total_bills=("ilga_all_bills_count", "sum"),
        primary_bills=("ilga_primary_bills_count", "sum"),
        cosponsored_bills=("ilga_cosponsored_bills_count", "sum"),
        sessions=("general_assembly", "nunique")
    )
    .sort_values("total_bills", ascending=False)
    .head(20)
)

chart7 = alt.Chart(top_women).mark_bar(cornerRadiusTopRight=5, cornerRadiusBottomRight=5).encode(
    y=alt.Y("ilga_name:N", sort="-x", title="Legislator"),
    x=alt.X("total_bills:Q", title="Total Bills Across Scraped General Assemblies"),
    color=alt.value("#CDB4DB"),
    tooltip=[
        "ilga_name",
        "total_bills",
        "primary_bills",
        "cosponsored_bills",
        "sessions"
    ]
).properties(
    title="Top 20 Illinois Congresswoman by Scraped Legislative Activity",
    width=750,
    height=500
)

style_chart(chart7)

alt.Chart(...)

In [23]:
top_heatmap_names = (
    df.groupby("ilga_name")["ilga_all_bills_count"]
    .sum()
    .sort_values(ascending=False)
    .head(30)
    .index
)

heatmap_df = df[df["ilga_name"].isin(top_heatmap_names)].copy()

chart8 = alt.Chart(heatmap_df).mark_rect().encode(
    x=alt.X("general_assembly:O", title="General Assembly"),
    y=alt.Y("ilga_name:N", title="Legislator", sort=list(top_heatmap_names)),
    color=alt.Color(
        "ilga_all_bills_count:Q",
        title="Bill Count",
        scale=alt.Scale(scheme="purpleblue")
    ),
    tooltip=[
        "ilga_name",
        "chamber",
        "general_assembly",
        "ilga_all_bills_count",
        "ilga_primary_bills_count",
        "ilga_cosponsored_bills_count"
    ]
).properties(
    title="Legislative Productivity Across General Assemblies",
    width=750,
    height=600
)

style_chart(chart8)

alt.Chart(...)